In [ ]:
%sql
-- dq_scorecard

-- One event log (a delta table) per pipeline
-- One pipeline - more runs
-- One run - more events
-- One evnet - one row in the table

In [ ]:
%sql
CREATE OR REPLACE VIEW insurance.medallion_dev.v_dq_scorecard AS

-- we get the full event log delta table. 
-- dim_date is used for identifying the pipeline to whom the event_log belongs to 
WITH events AS (
    SELECT *
    FROM event_log(TABLE(insurance.medallion_dev.dim_date))
),

-- keeps only the updates that finished successfully
latest_successful_update AS (
    SELECT origin.update_id
    FROM events
    WHERE event_type = 'update_progress' -- keeps the events referring to the state of the whole pipeline
      AND details:update_progress:state = 'COMPLETED' -- only the updates that completed successfully
    ORDER BY timestamp DESC
    LIMIT 1
),

expectations AS (
    SELECT
        event.timestamp AS run_ts,
        event.origin.flow_name,
        expectation
    FROM events AS event
    JOIN latest_successful_update AS latest
        ON event.origin.update_id = latest.update_id
    LATERAL VIEW explode(
        from_json(
            CAST(
                event.details:flow_progress:data_quality:expectations
                AS STRING
            ),
            'array<struct<
                name:string,
                dataset:string,
                passed_records:bigint,
                failed_records:bigint
            >>'
        )
    ) AS expectation
    WHERE event.event_type = 'flow_progress'
)

SELECT table_name, passed, quarantined, quarantine_rate, run_ts
FROM (
    SELECT
        CASE flow_name
            WHEN 'prepare_silver_customer'   THEN 'silver_customers'
            WHEN 'prepare_silver_policies'   THEN 'silver_policies'
            WHEN 'prepare_silver_claims'     THEN 'silver_claims'
            WHEN 'prepare_silver_payments'   THEN 'silver_payments'
            WHEN 'prepare_silver_telematics' THEN 'silver_telematics'
        END AS table_name,
        expectation.passed_records AS passed,
        expectation.failed_records AS quarantined,
        expectation.failed_records
            / NULLIF(
                expectation.passed_records + expectation.failed_records,
                0
            ) AS quarantine_rate,
        run_ts,
        ROW_NUMBER() OVER (
            PARTITION BY flow_name
            ORDER BY run_ts DESC
        ) AS row_number
    FROM expectations
    WHERE expectation.name = 'valid_record'
)
WHERE row_number = 1;